# aging_transformation

> 对应代码：`EconometricsCode/code_in_notes/Chap.8/aging_transformation.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.8`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.8")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们读入跨国宏观数据，并生成人均 GDP 的对数及其平方项。把水平项和平方项同时放入回归，是为了刻画制造业就业份额随经济发展水平变化的非线性（倒 U 形或 U 形）关系。

In [ ]:
%%stata
use ../datasets/aging_transformation_macro.dta, clear
// 不
gen log_gdp_pc  = log(gdp_pc)
gen log_gdp_pc2 = log_gdp_pc^2

我们首先估计一个含平方项的双向固定效应模型：被解释变量是制造业就业份额，解释变量包括对数人均 GDP 及其平方项，并控制 65 岁以上人口占比（老龄化程度）。reghdfe 的 absorb(code) 吸收国家固定效应，控制各国不随时间变化的异质性；vce(cl code) 在国家层面聚类标准误，允许同一国家内误差相关。

In [ ]:
%%stata
reghdfe hw_share_man log_gdp_pc log_gdp_pc2 share_65plus, a(code) vce(cl code)

接下来我们做一个去平均（demean）变换：先计算对数人均 GDP 的样本均值，再生成离差的平方。这样变换后，一次项系数的经济含义变为在样本均值处（而非原点处）的边际效应，解释更方便，且水平项与平方项之间的共线性也会下降。

In [ ]:
%%stata
egen mean_log_gdp_pc = mean(log_gdp_pc)
gen dm_log_gdp_pc2 = (log_gdp_pc-mean_log_gdp_pc)^2

我们用去平均后的平方项重新回归。两个设定的拟合值完全等价，但此时一次项系数直接给出经济发展处于样本平均水平时制造业份额的边际效应，而平方项系数在两种设定下完全相同。

In [ ]:
%%stata
reghdfe hw_share_man log_gdp_pc dm_log_gdp_pc2 share_65plus, a(code) vce(cl code)